# ZS601 2DGS LiDAR B → C on L4
Checkpoint-aware serial runner. It resumes a partial run, reruns postprocessing after a completed 150k training, and starts C only after B is complete.


In [ ]:
# ZS601 LiDAR B, then C on one L4 runtime
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, sys, json, time

os.environ['TORCH_CUDA_ARCH_LIST'] = '8.9'
subprocess.run(['nvidia-smi'], check=True)
print('Python:', sys.version, flush=True)

repo = Path('/content/ZS601_3DGS')
if not repo.exists():
    subprocess.run([
        'git','clone','--recursive','--branch','2dgs-zs601-mask-init',
        'https://github.com/VISjudy/ZS601_3DGS.git',str(repo)
    ], check=True)
else:
    subprocess.run(['git','-C',str(repo),'fetch','origin','2dgs-zs601-mask-init'], check=True)
    subprocess.run(['git','-C',str(repo),'checkout','2dgs-zs601-mask-init'], check=True)
    subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin','2dgs-zs601-mask-init'], check=True)
    subprocess.run(['git','-C',str(repo),'submodule','update','--init','--recursive'], check=True)
source = repo/'2d-gaussian-splattingWithMask'
launcher = source/'colab/run_parallel_experiment.py'
subprocess.run([sys.executable,'-m','py_compile',str(launcher)], check=True)

drive_root = Path('/content/drive/MyDrive/LCCDataset/zs601_output')
prefixes = {
    'b': 'zs601_2dgs_B_lidar_parallel_',
    'c': 'zs601_2dgs_C_lidar_distortion_parallel_',
}

def latest_run(mode):
    runs = [p for p in drive_root.glob(prefixes[mode] + '*') if p.is_dir()]
    return max(runs, key=lambda p: p.stat().st_mtime) if runs else None

def read_status(run_dir):
    try:
        return json.loads((run_dir/'status.json').read_text())
    except Exception:
        return {}

def run_mode(mode):
    run_dir = latest_run(mode)
    args = [sys.executable, '-u', str(launcher), '--mode', mode]
    if run_dir:
        status = read_status(run_dir)
        stage = status.get('stage')
        checkpoints = sorted(
            (run_dir/'model').glob('chkpnt*.pth'),
            key=lambda p: int(p.stem.replace('chkpnt', '')),
        )
        cp150 = run_dir/'model/chkpnt150000.pth'
        print(f'LATEST_{mode.upper()}={run_dir} stage={stage}', flush=True)
        if stage == 'complete':
            print(f'{mode.upper()} already complete; skipping.', flush=True)
            return
        if cp150.exists():
            print(f'{mode.upper()} training reached 150k; rerunning postprocess only.', flush=True)
            args += ['--resume-output', str(run_dir), '--postprocess-only']
        elif checkpoints:
            print(f'{mode.upper()} resuming training from {checkpoints[-1].name}.', flush=True)
            args += ['--resume-output', str(run_dir), '--resume-training']
        else:
            print(f'{mode.upper()} has no usable checkpoint; starting a fresh run.', flush=True)
    rc = subprocess.call(args, cwd=str(source))
    print(f'{mode.upper()}_RETURN_CODE={rc}', flush=True)
    if rc != 0:
        raise RuntimeError(f'{mode.upper()} failed. Later stages are intentionally paused.')

print('SERIAL PLAN: recover/finish B -> C (LiDAR, distortion=1000)', flush=True)
for mode in ('b', 'c'):
    print(f'===== START {mode.upper()} =====', flush=True)
    run_mode(mode)
print('B AND C COMPLETE', flush=True)
